# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors: Exploration with `mlcroissant`
This notebook provides a practical walk-through for loading, exploring, and analyzing the [FAIR² dataset on second primary colorectal cancer](https://doi.org/10.71728/senscience.qs2f-h81p) using the `mlcroissant` library. 

### Dataset Source
The dataset source is provided via a Croissant schema URL. All dataset entities (record sets, fields, columns) are referenced by their `@id`, ensuring clarity and reproducibility.

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = "https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json"

# Load the dataset with Croissant schema
dataset = mlc.Dataset(croissant_url)

# Access metadata safely (as object, not a dict!)
meta = dataset.metadata
print(f"Dataset: {meta.name}\n\n{meta.description}\n")

## 2. Data Overview
Let's review the available record sets, their associated fields and column `@id`s. All IDs shown are from the schema and are used to reference entities throughout the notebook.

In [ ]:
# List all available record sets with their `@id`, name, and fields’ IDs

for rs in dataset.record_sets:
    print(f"- Record Set @id: {rs['@id']}")
    print(f"  Name: {getattr(rs, 'name', '(no name)')}")
    if hasattr(rs, 'fields') and rs.fields:
        print("  Field @id\'s:")
        for fld in rs.fields:
            print(f"    - {fld['@id']}")
    if hasattr(rs, 'columns') and rs.columns:
        print("  Column @id\'s:")
        for col in rs.columns:
            print(f"    - {col['@id']}")
    print()

## 3. Data Extraction
We'll load the records for the main tabular dataset. Use the record set and field `@id`s discovered above (see previous cell output).

In this dataset, the main record set is typically a clinical-table (since it's a tabular clinical data resource). We'll load all record sets (in practice, often there is one).

In [ ]:
# Identify available record set IDs
record_set_ids = [rs['@id'] for rs in dataset.record_sets]
print("Available Record Sets IDs:")
for rid in record_set_ids:
    print(f"- {rid}")

# Load all records for each record set into DataFrames
dataframes = {}
for record_set_id in record_set_ids:
    print(f"\nLoading data for Record Set: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Fields/Columns available: {list(df.columns)}")
        display(df.head())
    else:
        print("No records found.")

# For further analysis, pick the first record set (if available)
if dataframes:
    main_record_set_id = list(dataframes.keys())[0]
    print(f"Selected for downstream analysis: {main_record_set_id}")
else:
    main_record_set_id = None
    print("No tabular record sets could be loaded.")

## 4. Exploratory Data Analysis (EDA)
Apply standard EDA steps: filtering, normalization, and grouping. All fields are referenced by Croissant `@id`.

**Note**: You must use the actual field/column `@id` from the loaded DataFrame. Adjust the `numeric_field_id` and `group_field_id` variables as appropriate after inspecting your record set in the previous section.

In [ ]:
# Example: Let's perform EDA if data is present
if main_record_set_id:
    df = dataframes[main_record_set_id]
    print(f"Working with record set: {main_record_set_id}")

    # Choose a numeric field `@id` (adjust as appropriate after inspecting the columns!)
    # Example IDs to check: '@id' might be like 'https://api.app.sen.science/frontiers/7862866/field:age'
    print("Available columns:", list(df.columns))
    # Try to guess likely numeric column, e.g. age, interval, size
    likely_numeric_fields = [col for col in df.columns if any(s in col.lower() for s in ['age', 'interval', 'size'])]
    numeric_field_id = likely_numeric_fields[0] if likely_numeric_fields else df.columns[0]

    # Filtering: threshold = 50 (for age as an example)
    threshold = 50
    filtered_df = df[df[numeric_field_id].astype(float) > threshold].copy()
    print(f"\nFiltered records with {numeric_field_id} > {threshold}:")
    display(filtered_df.head())

    # Normalization: z-score
    col_norm = f"{numeric_field_id}_normalized"
    filtered_df[col_norm] = (filtered_df[numeric_field_id].astype(float) - filtered_df[numeric_field_id].astype(float).mean()) / filtered_df[numeric_field_id].astype(float).std()
    print(f"\nFirst five normalized values for {numeric_field_id}:")
    display(filtered_df[[numeric_field_id, col_norm]].head())

    # Group by a likely categorical field, e.g. sex, tumor_location, msi_status
    likely_group_fields = [col for col in df.columns if any(s in col.lower() for s in ['sex', 'gender', 'msi', 'location'])]
    group_field_id = likely_group_fields[0] if likely_group_fields else df.columns[1]
    if group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame()
        print(f"\nGrouped mean {numeric_field_id} by {group_field_id}:")
        display(grouped_df)
else:
    print("No tabular data available for EDA.")

## 5. Visualization
Visualizing the distribution of a numeric variable and the grouped statistics using matplotlib and seaborn.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_record_set_id and not df.empty:
    plt.figure(figsize=(7,4))
    sns.histplot(df[numeric_field_id].astype(float), kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.tight_layout()
    plt.show()

    if group_field_id in df.columns:
        plt.figure(figsize=(7,4))
        sns.boxplot(x=df[group_field_id], y=df[numeric_field_id].astype(float))
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.tight_layout()
        plt.show()

## 6. Conclusion
We have demonstrated how to load, inspect, and analyze a real-world clinical dataset with a FAIR Croissant schema using `mlcroissant`. 
- All entities were referenced by their Croissant `@id` for transparency and reproducibility.
- We loaded all available tabular record sets, explored their fields, and conducted basic EDA—including normalization and grouping by key variables.
- Data visualizations gave an overview of data distributions and group statistics.

Proceed to develop further analyses or model-building using the loaded DataFrames as needed!